In [ ]:
import os
import random
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from collections import Counter
from tqdm import tqdm
import matplotlib.pyplot as plt
import cv2

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

def seed_worker(worker_id):
    worker_seed = 42 + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
DATA_ROOT = "/kaggle/input/bach-breast-cancer-histology-images"
TRAIN_DIR = os.path.join(
    DATA_ROOT,
    "ICIAR2018_BACH_Challenge",
    "ICIAR2018_BACH_Challenge",
    "Photos"
)

CLASS_NAMES = ["Normal", "Benign", "InSitu", "Invasive"]
class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}

def load_samples(root):
    samples = []
    for cls in CLASS_NAMES:
        cls_dir = os.path.join(root, cls)
        for f in os.listdir(cls_dir):
            if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif")):
                samples.append((os.path.join(cls_dir, f), class_to_idx[cls]))
    return samples

all_samples = load_samples(TRAIN_DIR)

In [ ]:
train_samples, temp_samples = train_test_split(
    all_samples,
    test_size=0.2,
    stratify=[s[1] for s in all_samples],
    random_state=42
)

val_samples, test_samples = train_test_split(
    temp_samples,
    test_size=0.5,
    stratify=[s[1] for s in temp_samples],
    random_state=42
)

print("Train:", len(train_samples))
print("Val:", len(val_samples))
print("Test:", len(test_samples))

In [ ]:
class BACHDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        return img, label


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((230,230)),
    transforms.RandomRotation(1),
    transforms.RandomResizedCrop(224, scale=(0.7,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

In [ ]:
train_targets = [s[1] for s in train_samples]
class_count = Counter(train_targets)

weights = 1. / torch.tensor(
    [class_count[i] for i in range(len(CLASS_NAMES))],
    dtype=torch.float
)

samples_weights = torch.tensor([weights[t] for t in train_targets])

sampler = WeightedRandomSampler(
    samples_weights,
    len(samples_weights),
    replacement=True,
    generator=g
)

train_loader = DataLoader(
    BACHDataset(train_samples, train_tf),
    batch_size=32,
    sampler=sampler,
    num_workers=2,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=g
)

val_loader = DataLoader(
    BACHDataset(val_samples, val_tf),
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=g
)

test_loader = DataLoader(
    BACHDataset(test_samples, val_tf),
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=g
)


In [ ]:
model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
model.classifier[2] = nn.Linear(model.classifier[2].in_features, 4)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.15)

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-2
)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-4,
    steps_per_epoch=len(train_loader),
    epochs=100,
    pct_start=0.3
)

In [ ]:
def calculate_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    return acc, p, r, f

In [ ]:
EPOCHS = 100
PATIENCE = 25

train_losses = []
val_losses = []
train_accs = []
val_accs = []

best_f1 = 0
best_epoch = 0
best_state = None
best_metrics = None
best_pred = None
patience_counter = 0

print(f"{'Epoch':^6} | {'LR':^9} | {'Train Loss':^10} | {'Val Loss':^9} | {'Val Acc':^9} | {'Val Prec':^9} | {'Val Rec':^9} | {'Val F1':^9} | Status")
print("-"*120)

for epoch in range(1, EPOCHS+1):

    # ================= TRAIN =================
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    current_lr = scheduler.get_last_lr()[0]

    for imgs, labels in tqdm(train_loader, leave=False):
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        loss.backward()

        # 🔥 Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)

        optimizer.step()
        scheduler.step()  

        train_loss += loss.item()
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss /= len(train_loader)
    train_acc = correct / total

    # ================= VALIDATION =================
    model.eval()
    y_true, y_pred = [], []
    val_loss = 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)

            loss = criterion(outputs, labels.to(device))
            val_loss += loss.item()

            preds = outputs.argmax(1)
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())

    val_loss /= len(val_loader)
    val_acc, val_prec, val_rec, val_f1 = calculate_metrics(y_true, y_pred)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    status = ""

    # ================= SAVE BEST =================
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        best_metrics = (val_acc, val_prec, val_rec, val_f1)
        best_pred = (y_true.copy(), y_pred.copy())
        patience_counter = 0
        status = "BEST"
    else:
        patience_counter += 1

    print(f"{epoch:^6} | {current_lr:^9.2e} | {train_loss:^10.4f} | {val_loss:^9.4f} | "
          f"{val_acc*100:^8.2f}% | {val_prec*100:^8.2f}% | "
          f"{val_rec*100:^8.2f}% | {val_f1*100:^8.2f}% | {status}")

    # ================= EARLY STOP =================
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

In [ ]:
# ================== TEST ==================
model.load_state_dict(best_state)
model.eval()

y_true_test, y_pred_test = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = torch.argmax(outputs, dim=1)

        y_true_test.extend(labels.numpy())
        y_pred_test.extend(preds.cpu().numpy())

# ================== METRICS ==================
test_acc, test_prec, test_rec, test_f1 = calculate_metrics(
    y_true_test, y_pred_test
)

print("\n" + "="*50)
print("FINAL TEST RESULT")
print("="*50)
print(f"Accuracy  : {test_acc*100:.2f}%")
print(f"Precision : {test_prec*100:.2f}%")
print(f"Recall    : {test_rec*100:.2f}%")
print(f"F1-score  : {test_f1*100:.2f}%")

# ================== REPORT ==================
print("\nClassification Report (Test):")
print(classification_report(
    y_true_test,
    y_pred_test,
    target_names=CLASS_NAMES
))

# ================== CONFUSION MATRIX ==================
cm = confusion_matrix(y_true_test, y_pred_test)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES
)

disp.plot(xticks_rotation=45)
plt.title("Confusion Matrix (Test Set)")
plt.grid(False)
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.title("Train vs Val Loss")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(6,4))
plt.plot(train_accs, label="Train Accuracy")
plt.plot(val_accs, label="Val Accuracy")
plt.title("Accuracy")
plt.legend()
plt.grid()
plt.show()

In [ ]:
features, gradients = [], []

def forward_hook(m,i,o):
    features.clear()
    features.append(o)

def backward_hook(m,gi,go):
    gradients.clear()
    gradients.append(go[0])

model.features[-1].register_forward_hook(forward_hook)
model.features[-1].register_full_backward_hook(backward_hook)

def grad_cam_single(img, target_class):
    img = img.unsqueeze(0).to(device)

    output = model(img)

    model.zero_grad()
    output[0, target_class].backward(retain_graph=True)

    fmap = features[0]
    grad = gradients[0]

    weights = grad.mean(dim=(2,3), keepdim=True)
    cam = (weights * fmap).sum(dim=1).squeeze()

    cam = cam.detach().cpu().numpy()
    cam = np.maximum(cam,0)

    if cam.max() != 0:
        cam = cam / cam.max()

    cam = cv2.resize(cam,(224,224))
    return cam

def denormalize(img):
    img = img.clone()
    for t, m, s in zip(img, IMAGENET_MEAN, IMAGENET_STD):
        t.mul_(s).add_(m)
    return img.permute(1,2,0).numpy()

def save_three_images(img, label, save_dir="outputs"):
    os.makedirs(save_dir, exist_ok=True)

    model.eval()
    img_input = img.unsqueeze(0).to(device)

    output = model(img_input)
    pred_class = torch.argmax(output, dim=1).item()

    img_np = denormalize(img)

    # 1. ORIGINAL
    plt.figure(figsize=(5,5))
    plt.imshow(img_np)
    plt.title(f"Ground Truth: {CLASS_NAMES[label]}")
    plt.axis('off')
    plt.savefig(f"{save_dir}/original.png", bbox_inches='tight')
    plt.close()

    # 2. GRADCAM GT
    cam_gt = grad_cam_single(img, label)

    plt.figure(figsize=(5,5))
    plt.imshow(img_np)
    plt.imshow(cam_gt, cmap='jet', alpha=0.5)
    plt.title(f"Grad-CAM (GT): {CLASS_NAMES[label]}")
    plt.axis('off')
    plt.savefig(f"{save_dir}/gradcam_gt.png", bbox_inches='tight')
    plt.close()

    # 3. ALL CLASS (4 class BACH)
    cams_all = [grad_cam_single(img, i) for i in range(len(CLASS_NAMES))]

    plt.figure(figsize=(10,8))

    for i in range(len(CLASS_NAMES)):
        plt.subplot(2,2,i+1)

        plt.imshow(img_np)
        plt.imshow(cams_all[i], cmap='jet', alpha=0.5)

        if i == label:
            plt.title(f"{CLASS_NAMES[i]} (GT)", color='red')
        elif i == pred_class:
            plt.title(f"{CLASS_NAMES[i]} (Pred)")
        else:
            plt.title(CLASS_NAMES[i])

        plt.axis('off')

    status = "Correct" if pred_class == label else "Wrong"

    plt.suptitle(
        f"GT: {CLASS_NAMES[label]} | Pred: {CLASS_NAMES[pred_class]} ({status})",
        fontsize=14
    )

    plt.tight_layout()
    plt.savefig(f"{save_dir}/gradcam_all.png", bbox_inches='tight')
    plt.close()

    print(f"Saved 3 images to: {save_dir}")

img, label = BACHDataset(test_samples, val_tf)[0]
save_three_images(img, label)